# Different Efficient Attention Variants
### Problem Statement
Standard scaled dot-product attention materializes an `n x n` score matrix, so memory and compute grow as `O(n^2)`. Efficient attention methods reduce this cost in different ways, and interviewers often care more about the tradeoff than one specific implementation.

Your goal is to understand several common families and implement two small variants from scratch:

1. **Sliding-window / local attention**: each token attends only to nearby tokens. This is useful for long contexts where most interactions are local.
2. **Linear attention**: replace softmax attention with a kernel feature map so attention can be computed with prefix sums in causal settings.

---

### Efficient Attention Families

| Method | Main idea | Exact softmax? | Typical benefit | Main tradeoff |
|---|---|---:|---|---|
| FlashAttention | Tile Q/K/V through SRAM and avoid storing the full score matrix | Yes | Lower memory, faster exact attention | Kernel-level implementation complexity |
| Sliding-window / local | Restrict each query to a fixed neighborhood | No | `O(n * window)` instead of `O(n^2)` | Long-range dependencies need extra mechanism |
| Block sparse / global tokens | Attend to selected blocks or special global tokens | Usually no | Better long-context scaling | Pattern design matters |
| MQA / GQA | Share K/V heads across query heads | Yes for same attention formula | Smaller KV cache and faster decoding | Less K/V head capacity |
| Linear attention | Use kernel features: `softmax(QK^T)V ≈ φ(Q)(φ(K)^T V)` | No | Can be `O(n * d^2)` or streaming causal | Approximation quality varies |
| Low-rank / Nyström / Linformer | Project or approximate the attention matrix | No | Reduces sequence dimension cost | Approximation can hurt tasks needing precise recall |

---

### Interview Mental Model

- If the method returns the same result as softmax attention but uses less memory, it is an **exact systems optimization**. FlashAttention is the canonical example.
- If the method changes who can attend to whom, it is a **sparsity pattern**. Sliding-window and block-sparse attention are examples.
- If the method changes the math of softmax attention, it is an **approximation**. Linear and low-rank attention are examples.
- For autoregressive LLM serving, efficient attention often means **KV-cache efficiency**: MQA/GQA reduce cache size, while FlashAttention accelerates prefill and attention kernels.

---

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

## Baseline: Full Scaled Dot-Product Attention

This helper is the reference implementation. It materializes all pairwise query-key scores with shape `(batch, heads, seq_len, seq_len)`.

In [ ]:
def full_attention(q, k, v, causal=False):
    """Standard scaled dot-product attention.

    q, k, v: (batch, heads, seq_len, d_head)
    """
    d_head = q.size(-1)
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_head)

    if causal:
        seq_len = q.size(-2)
        mask = torch.triu(torch.ones(seq_len, seq_len, device=q.device), diagonal=1).bool()
        scores = scores.masked_fill(mask, float("-inf"))

    weights = F.softmax(scores, dim=-1)
    return torch.matmul(weights, v), weights

## Variant 1: Sliding-Window Attention

Each token attends only to tokens within a fixed radius. With `window_size = w`, each query sees at most `2w + 1` keys in bidirectional mode, or at most `w + 1` keys in causal mode.

This implementation still builds a dense mask for clarity. Production kernels avoid doing work for masked positions, giving the real `O(n * w)` benefit.

In [ ]:
def sliding_window_attention(q, k, v, window_size, causal=False):
    """Sliding-window attention with an explicit mask for readability.

    q, k, v: (batch, heads, seq_len, d_head)
    window_size: number of neighboring positions visible on each side
    """
    batch, heads, seq_len, d_head = q.shape
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_head)

    positions = torch.arange(seq_len, device=q.device)
    distance = positions[:, None] - positions[None, :]

    if causal:
        allowed = (distance >= 0) & (distance <= window_size)
    else:
        allowed = distance.abs() <= window_size

    scores = scores.masked_fill(~allowed, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    return torch.matmul(weights, v), weights

## Variant 2: Causal Linear Attention

Softmax attention cannot be decomposed as prefix sums directly. Linear attention uses a positive feature map `φ(x)` and approximates attention as:

`Attention(Q, K, V)_t = φ(q_t)^T sum_{i<=t}(φ(k_i) v_i^T) / φ(q_t)^T sum_{i<=t} φ(k_i)`

Because the sums are prefix sums, causal inference can stream through the sequence without storing an `n x n` score matrix.

In [ ]:
def elu_feature_map(x):
    """Positive feature map often used in simple linear-attention examples."""
    return F.elu(x) + 1.0


def causal_linear_attention(q, k, v, eps=1e-6):
    """Causal linear attention using prefix sums.

    q, k, v: (batch, heads, seq_len, d_head)
    Returns: (batch, heads, seq_len, d_head)
    """
    q_phi = elu_feature_map(q)
    k_phi = elu_feature_map(k)

    # For each position t, keep sum_{i<=t} phi(k_i) outer v_i.
    kv = torch.einsum("bhnd,bhne->bhnde", k_phi, v)
    kv_prefix = kv.cumsum(dim=2)
    k_prefix = k_phi.cumsum(dim=2)

    numerator = torch.einsum("bhnd,bhnde->bhne", q_phi, kv_prefix)
    denominator = torch.einsum("bhnd,bhnd->bhn", q_phi, k_prefix).unsqueeze(-1)
    return numerator / (denominator + eps)

## Quick Validation

The sliding-window output should match full attention when the window covers the whole sequence. Linear attention has the same shape as full causal attention, but it is intentionally not numerically identical because it changes the attention kernel.

In [ ]:
torch.manual_seed(42)
batch_size = 2
num_heads = 3
seq_len = 6
d_head = 4

q = torch.randn(batch_size, num_heads, seq_len, d_head)
k = torch.randn(batch_size, num_heads, seq_len, d_head)
v = torch.randn(batch_size, num_heads, seq_len, d_head)

full_out, full_weights = full_attention(q, k, v, causal=False)
wide_local_out, wide_local_weights = sliding_window_attention(
    q, k, v, window_size=seq_len - 1, causal=False
)

causal_full_out, _ = full_attention(q, k, v, causal=True)
linear_out = causal_linear_attention(q, k, v)

print("Sliding window equals full attention when window covers all tokens:", torch.allclose(full_out, wide_local_out, atol=1e-6))
print("Full attention output shape:       ", tuple(full_out.shape))
print("Causal linear attention shape:    ", tuple(linear_out.shape))
print("Linear attention differs from softmax:", not torch.allclose(causal_full_out, linear_out, atol=1e-3))

## Practical Selection Guide

- Use **FlashAttention** when you want exact attention but better GPU memory behavior and speed.
- Use **MQA/GQA** when decoder KV-cache memory or decode bandwidth is the bottleneck.
- Use **sliding-window or block-sparse attention** when the task has locality or a designed global-token pattern.
- Use **linear or low-rank attention** when approximate long-context scaling is acceptable and exact retrieval is less critical.
- For interviews, always say whether the method is exact or approximate, and whether it reduces training prefill cost, decoding cache cost, or both.